In [1]:
# Cell 1
# ============================================================
# 02_sql_analysis.ipynb
# Olist Supply Chain Risk Intelligence
# ============================================================
# This notebook:
# 1. Connects to BigQuery
# 2. Runs analytical SQL queries on the master table
# 3. Computes KPIs, delay by state, monthly trends,
#    category breakdown, and stockout risk
# 4. Saves results as CSVs and back to BigQuery
# Prerequisites: 01_data_ingestion.ipynb must be run first
# ============================================================

In [2]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import pandas_gbq
from google.cloud import bigquery
from google.oauth2 import service_account

print("Libraries loaded ✅")

Libraries loaded ✅


In [3]:
# Cell 3 — Configuration
KEY_PATH   = "bq_key.json"   # update path if different

PROJECT_ID = "supply-chain-analytics-500607"
DATASET_ID = "olist"
TABLE_ID   = "orders_master"
TABLE      = f"`{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`"

credentials = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

def run_query(sql):
    return client.query(sql).to_dataframe()

def save(df, filename, bq_table):
    df.to_csv(f"../data/{filename}", index=False)
    pandas_gbq.to_gbq(
        df,
        destination_table=f"{DATASET_ID}.{bq_table}",
        project_id=PROJECT_ID,
        credentials=credentials,
        if_exists="replace",
        progress_bar=False
    )
    print(f"Saved: data/{filename} + BigQuery {DATASET_ID}.{bq_table} ✅")

print(f"Connected to BigQuery: {PROJECT_ID}.{DATASET_ID}")

Connected to BigQuery: supply-chain-analytics-500607.olist


In [4]:
# Cell 4 — Verify master table exists
check = run_query(f"""
    SELECT COUNT(*) AS total_rows
    FROM {TABLE}
""")
print(f"Master table rows: {check['total_rows'][0]:,}")
print("Master table confirmed ✅")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Master table rows: 110,189
Master table confirmed ✅


In [5]:
# Cell 5 — Query 1: Overall KPIs
kpis = run_query(f"""
    SELECT
        COUNT(DISTINCT order_id)                            AS total_orders,
        COUNT(DISTINCT seller_id)                           AS total_sellers,
        COUNT(DISTINCT customer_id)                         AS total_customers,
        COUNT(DISTINCT product_id)                          AS total_products,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_order_pct,
        ROUND(AVG(shipping_duration_days), 2)               AS avg_shipping_days,
        ROUND(SUM(revenue_at_risk), 2)                      AS total_revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value
    FROM {TABLE}
    WHERE delivery_delay_days IS NOT NULL
""")

print("=" * 50)
print("OVERALL KPIs")
print("=" * 50)
for col in kpis.columns:
    print(f"  {col:<30} : {kpis[col][0]}")

OVERALL KPIs
  total_orders                   : 96470
  total_sellers                  : 2970
  total_customers                : 96470
  total_products                 : 32214
  avg_delay_days                 : -12.03
  late_order_pct                 : 6.59
  avg_shipping_days              : 12.01
  total_revenue_at_risk          : 1368508.71
  avg_review_score               : 4.05
  avg_order_value                : 179.47


In [6]:
# Cell 6 — Query 2: Delivery delay by customer state
delay_by_customer_state = run_query(f"""
    SELECT
        customer_state,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score
    FROM {TABLE}
    WHERE customer_state IS NOT NULL
      AND delivery_delay_days IS NOT NULL
    GROUP BY customer_state
    ORDER BY late_pct DESC
""")

print(f"States analyzed: {len(delay_by_customer_state)}")
print(delay_by_customer_state.head(10).to_string())
save(delay_by_customer_state, "delay_by_customer_state.csv", "delay_by_customer_state")

States analyzed: 27
  customer_state  total_orders  avg_delay_days  late_pct  revenue_at_risk  avg_review_score
0             AL           397           -8.74     20.84         23950.73              3.79
1             MA           717           -9.91     18.00         40211.20              3.73
2             SE           335          -10.00     16.27         16452.85              3.88
3             CE          1279          -11.10     13.60         46038.63              3.86
4             PI           476          -11.53     13.58         15037.93              3.92
5             BA          3256          -10.98     11.89         84705.12              3.83
6             RJ         12350          -12.01     11.62        291177.54              3.83
7             PA           946          -14.25     11.29         29859.26              3.77
8             RR            41          -18.33     10.87          1454.39              3.89
9             PB           517          -13.04     10.75    

In [7]:
# Cell 7 — Query 3: Delivery delay by seller state
delay_by_seller_state = run_query(f"""
    SELECT
        seller_state,
        COUNT(DISTINCT order_id)                            AS total_orders,
        COUNT(DISTINCT seller_id)                           AS total_sellers,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk
    FROM {TABLE}
    WHERE seller_state IS NOT NULL
      AND delivery_delay_days IS NOT NULL
    GROUP BY seller_state
    ORDER BY late_pct DESC
""")

print(f"Seller states analyzed: {len(delay_by_seller_state)}")
print(delay_by_seller_state.head(10).to_string())
save(delay_by_seller_state, "delay_by_seller_state.csv", "delay_by_seller_state")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Seller states analyzed: 22
  seller_state  total_orders  total_sellers  avg_delay_days  late_pct  revenue_at_risk
0           AM             3              1            9.00     33.33           984.26
1           MA           389              1          -11.26     19.40          9911.40
2           RN            51              5          -13.48      7.14          2902.69
3           SP         68635           1769          -11.30      7.11        971089.56
4           RJ          4227            163          -12.48      6.92         64667.72
5           CE            87             12          -13.38      6.67           979.34
6           DF           808             30          -13.18      6.00          7738.88
7           MS            49              5          -17.38      6.00           618.66
8           ES           310             22          -13.36      5.77          6424.83
9           PR          7512            335          -14.23      5.30        104975.59
Saved: data/dela

In [8]:
# Cell 8 — Query 4: Monthly delivery trend
monthly_trend = run_query(f"""
    SELECT
        order_month,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score
    FROM {TABLE}
    WHERE order_month IS NOT NULL
      AND delivery_delay_days IS NOT NULL
    GROUP BY order_month
    HAVING total_orders >= 50
    ORDER BY order_month ASC
""")

print(f"Months analyzed: {len(monthly_trend)}")
print(monthly_trend.to_string())
save(monthly_trend, "monthly_trend.csv", "monthly_trend")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Months analyzed: 21
   order_month  total_orders  avg_delay_days  late_pct  revenue_at_risk  avg_review_score
0      2016-10           265          -36.83      0.64           198.86              3.85
1      2017-01           750          -27.81      2.63          3863.20              4.11
2      2017-02          1653          -19.18      3.12          8657.44              4.16
3      2017-03          2546          -12.44      4.69         24925.68              4.10
4      2017-04          2303          -13.12      6.23         34601.15              4.08
5      2017-05          3545          -13.39      3.20         22743.65              4.17
6      2017-06          3135          -12.72      2.95         29691.80              4.16
7      2017-07          3872          -12.63      3.01         30281.12              4.16
8      2017-08          4193          -13.36      2.77         21359.39              4.23
9      2017-09          4150          -11.44      4.39         37783.09         

In [9]:
# Cell 9 — Query 5: Delay by product category
delay_by_category = run_query(f"""
    SELECT
        product_category_name_english                       AS category,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value
    FROM {TABLE}
    WHERE product_category_name_english IS NOT NULL
      AND product_category_name_english != 'unknown'
      AND delivery_delay_days IS NOT NULL
    GROUP BY category
    HAVING total_orders >= 100
    ORDER BY late_pct DESC
    LIMIT 20
""")

print(f"Categories analyzed: {len(delay_by_category)}")
print(delay_by_category.to_string())
save(delay_by_category, "delay_by_category.csv", "delay_by_category")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Categories analyzed: 20
                     category  total_orders  avg_delay_days  late_pct  revenue_at_risk  avg_order_value
0                       audio           348          -10.15     11.60          7590.27           166.25
1          christmas_supplies           125          -12.05     10.00          6484.90           125.96
2     fashion_underwear_beach           117          -10.93      9.45           959.36            97.77
3                home_confort           392           -9.81      9.32          7010.70           195.09
4             books_technical           256          -11.31      7.98          1429.40            92.92
5            office_furniture          1254          -11.85      7.97         47412.35           381.37
6                        baby          2809          -11.65      7.68         44836.43           176.24
7                 electronics          2517          -11.14      7.59         21045.04            90.10
8               health_beauty          8

In [10]:
# Cell 10 — Query 6: Demand concentration risk (renamed from stockout)
demand_concentration = run_query(f"""
    SELECT
        product_category_name_english                       AS category,
        COUNT(DISTINCT order_id)                            AS order_volume,
        COUNT(DISTINCT seller_id)                           AS seller_count,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value,
        ROUND(COUNT(DISTINCT order_id) /
              (COUNT(DISTINCT seller_id) + 1), 2)           AS demand_per_seller,
        ROUND(
            COUNT(DISTINCT order_id) *
            AVG(total_payment_value) /
            (COUNT(DISTINCT seller_id) + 1), 2
        )                                                   AS demand_concentration_score
    FROM {TABLE}
    WHERE product_category_name_english IS NOT NULL
      AND product_category_name_english != 'unknown'
    GROUP BY category
    HAVING order_volume >= 50
    ORDER BY demand_concentration_score DESC
    LIMIT 30
""")

def concentration_tier(score):
    if score >= 50000:   return "Critical"
    elif score >= 20000: return "High"
    elif score >= 5000:  return "Medium"
    else:                return "Low"

demand_concentration["risk_label"] = demand_concentration[
    "demand_concentration_score"
].apply(concentration_tier)

print(f"Risk distribution:\n{demand_concentration['risk_label'].value_counts()}")
print(demand_concentration.head(10).to_string())
save(demand_concentration, "demand_concentration.csv", "demand_concentration")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Risk distribution:
risk_label
Low       26
Medium     3
High       1
Name: count, dtype: int64
                                category  order_volume  seller_count  avg_order_value  demand_per_seller  demand_concentration_score risk_label
0                              computers           177             9          1380.26              17.70                    24430.61       High
1                       office_furniture          1254            30           381.37              40.45                    15427.02     Medium
2                          watches_gifts          5493            95           236.82              57.22                    13550.46     Medium
3                         bed_bath_table          9272           189           154.54              48.80                     7541.72     Medium
4                  computers_accessories          6529           279           202.70              23.32                     4726.58        Low
5                           home_confort 

In [11]:
# Cell 11 — Query 7: Payment type analysis
payment_analysis = run_query(f"""
    SELECT
        payment_type,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days
    FROM {TABLE}
    WHERE payment_type IS NOT NULL
    GROUP BY payment_type
    ORDER BY total_orders DESC
""")

print(payment_analysis.to_string())
save(payment_analysis, "payment_analysis.csv", "payment_analysis")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  payment_type  total_orders  avg_order_value  late_pct  avg_delay_days
0  credit_card         73213           182.50      6.49          -12.24
1       boleto         19191           176.33      7.11          -11.29
2      voucher          2582           132.49      6.12          -11.87
3   debit_card          1483           149.20      5.63          -11.46
Saved: data/payment_analysis.csv + BigQuery olist.payment_analysis ✅


In [12]:
# Cell 12 — Query 8: Year over year comparison
yoy = run_query(f"""
    SELECT
        order_year,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score
    FROM {TABLE}
    WHERE order_year IS NOT NULL
      AND delivery_delay_days IS NOT NULL
    GROUP BY order_year
    ORDER BY order_year ASC
""")

print("Year over Year:")
print(yoy.to_string())
save(yoy, "yoy_comparison.csv", "yoy_comparison")

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Year over Year:
   order_year  total_orders  avg_delay_days  late_pct  revenue_at_risk  avg_review_score
0        2016           267          -36.09      1.58           198.86              3.83
1        2017         43426          -12.44      5.55        529100.51              4.07
2        2018         52777          -11.56      7.48        839209.34              4.03
Saved: data/yoy_comparison.csv + BigQuery olist.yoy_comparison ✅


In [13]:
# Cell 13 — Summary
print("=" * 55)
print("02_sql_analysis.ipynb — COMPLETE")
print("=" * 55)
print(f"\nQueries run:")
print(f"  1. Overall KPIs")
print(f"  2. Delay by customer state   ({len(delay_by_customer_state)} states)")
print(f"  3. Delay by seller state     ({len(delay_by_seller_state)} states)")
print(f"  4. Monthly trend             ({len(monthly_trend)} months)")
print(f"  5. Delay by category         ({len(delay_by_category)} categories)")
print(f"  6. Demand concentration risk ({len(demand_concentration)} categories)")
print(f"  7. Payment type analysis     ({len(payment_analysis)} types)")
print(f"  8. Year over year comparison ({len(yoy)} years)")
print(f"\nCSVs saved to /data:")
csvs = [
    "delay_by_customer_state.csv",
    "delay_by_seller_state.csv",
    "monthly_trend.csv",
    "delay_by_category.csv",
    "demand_concentration.csv",
    "payment_analysis.csv",
    "yoy_comparison.csv"
]
for c in csvs:
    print(f"  {c}")
print(f"\nAll tables also saved to BigQuery: {DATASET_ID}.*")
print(f"\nNext → run 03_ml_models.ipynb")

02_sql_analysis.ipynb — COMPLETE

Queries run:
  1. Overall KPIs
  2. Delay by customer state   (27 states)
  3. Delay by seller state     (22 states)
  4. Monthly trend             (21 months)
  5. Delay by category         (20 categories)
  6. Demand concentration risk (30 categories)
  7. Payment type analysis     (4 types)
  8. Year over year comparison (3 years)

CSVs saved to /data:
  delay_by_customer_state.csv
  delay_by_seller_state.csv
  monthly_trend.csv
  delay_by_category.csv
  demand_concentration.csv
  payment_analysis.csv
  yoy_comparison.csv

All tables also saved to BigQuery: olist.*

Next → run 03_ml_models.ipynb
